# 1. job_posting데이터 전처리

# 중복데이터 제거

In [1]:
# 원본 데이터 (307건)
#     ↓
# 중복 제거 (284건) ← 23건 제거
#     ↓
# 짧은 텍스트 필터링 (282건) ← 2건 제거
#     ↓
# ✅ 최종 데이터: 282건

In [4]:
import re
import pandas as pd

IN_PATH = "datas/raw/통합_AI세분류_7개직군_jobposting_0331.csv"
OUT_PATH = "datas/processed/job_postings_processed.csv"

df = pd.read_csv(IN_PATH)

# ====== 1) '?' 기준 split 방식으로 wantedAuthNo 추출 ======
def extract_wantedAuthNo(url):
    if pd.isna(url):
        return None
    
    url = str(url)
    
    # '?' 이후 쿼리 문자열 분리
    parts = url.split("?", 1)
    if len(parts) < 2:
        return None
    
    query_string = parts[1]  # wantedAuthNo=152033&infoTypeCd=OEW...
    
    # '&'로 분리
    params = query_string.split("&")
    
    for param in params:
        if param.startswith("wantedAuthNo="):
            return param.split("=", 1)[1]
    
    return None

df["job_posting_id"] = df["url"].apply(extract_wantedAuthNo)

# ====== 2) text 구성 ======
TEXT_COLS = ["주요업무", "자격요건", "우대사항", "직무"]

def clean_text(x):
    if pd.isna(x):
        return ""
    s = str(x)
    s = s.replace("\r\n", " ").replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\b(주요업무|자격요건|우대사항)\b\s*:?", " ", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s

df["text"] = (
    df[TEXT_COLS]
    .fillna("")
    .astype(str)
    .apply(lambda row: " ".join([clean_text(v) for v in row if clean_text(v)]), axis=1)
)

# ====== 3) 정리 ======
out = df.rename(
    columns={
        "공고제목": "title",
        "회사명": "company",
        "접수시작일": "date",
    }
)[["job_posting_id", "title", "company", "text", "date", "url"]]

out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("완료")

완료


# ncs 데이터 전처리

In [5]:
import re
import pandas as pd

DATA_BASE = "datas/raw"  # 데이터베이스 이름(필요 시 수정)
IN_PATH = f"{DATA_BASE}/ncs_info.csv"              # 필요 시 경로 수정
OUT_PATH = "datas/processed/ncs_roles_processed.csv"

df = pd.read_csv(IN_PATH)

# -----------------------------
# 1) 기본 품질 점검
# -----------------------------
required_cols = [
    "ncs_id", "large_category", "mid_category", "small_category",
    "job_role_name", "job_role_desc"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# ncs_id를 문자열로 통일(앞자리 0 등의 이슈 방지)
df["ncs_id"] = df["ncs_id"].astype(str).str.strip()

# -----------------------------
# 2) 공통 정규화 함수
# -----------------------------
def norm_space(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)

    # 줄바꿈/탭 정리
    s = s.replace("\r\n", " ").replace("\r", " ").replace("\n", " ").replace("\t", " ")

    # 불필요한 라벨 제거(있을 때만 제거)
    # 예: "직무정의:", "직무 정의 :", "정의 :" 같은 표현
    s = re.sub(r"\b(직무\s*정의|직무정의|정의)\b\s*:?\s*", " ", s)

    # 중복 공백 정리
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s

# 카테고리/이름/설명 정리
for c in ["large_category", "mid_category", "small_category", "job_role_name", "job_role_desc"]:
    df[c] = df[c].apply(norm_space)

# -----------------------------
# 3) 핵심: 모델 입력용 대표 텍스트 생성
#    - Top-k 유사도 비교에 사용할 텍스트
# -----------------------------
# 권장: 역할명 + 설명을 결합해 분별력 강화
df["role_text"] = (df["job_role_name"] + " | " + df["job_role_desc"]).str.strip()

# 계층 경로(분석/일반화용)
df["hierarchy_path"] = (
    df["large_category"] + " > " +
    df["mid_category"] + " > " +
    df["small_category"] + " > " +
    df["job_role_name"]
)

# -----------------------------
# 4) 필수 품질 게이트(중복/결측)
# -----------------------------
# ncs_id 중복 검사
dup_cnt = df["ncs_id"].duplicated().sum()
if dup_cnt > 0:
    dups = df[df["ncs_id"].duplicated(keep=False)].sort_values("ncs_id")
    raise ValueError(f"Duplicated ncs_id found: {dup_cnt}\n{dups[['ncs_id','job_role_name']].to_string(index=False)}")

# role_text가 너무 짧으면 비교 품질이 떨어짐(필요 시 기준 조정)
df["role_text_len"] = df["role_text"].str.len()
too_short = (df["role_text_len"] < 30).sum()

print(f"[CHECK] rows={len(df)}")
print(f"[CHECK] duplicated ncs_id={dup_cnt}")
print(f"[CHECK] role_text too short(<30 chars)={too_short}")

# -----------------------------
# 5) 저장(실험용 컬럼만 정리)
# -----------------------------
out = df[[
    "ncs_id",
    "large_category", "mid_category", "small_category",
    "job_role_name", "job_role_desc",
    "role_text", "hierarchy_path"
]]

out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"[OK] saved: {OUT_PATH}")

[CHECK] rows=7
[CHECK] duplicated ncs_id=0
[CHECK] role_text too short(<30 chars)=0
[OK] saved: datas/processed/ncs_roles_processed.csv
